In [1]:
#import and configuration
import numpy as np
import pandas as pd
import os

data_path = "../dataset/ml-1m"

ratings = pd.read_csv(
    os.path.join(data_path, "ratings.dat"),
    sep="::", engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    os.path.join(data_path, "movies.dat"),
    sep="::", engine="python",
    encoding="ISO-8859-1",
    names=["movie_id", "title", "genres"]
)

users = pd.read_csv(
    os.path.join(data_path, "users.dat"),
    sep="::", engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Users:", users.shape)

Ratings: (1000209, 4)
Movies: (3883, 3)
Users: (6040, 5)


In [2]:
#data preparing 
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["movie_id"] = ratings["movie_id"].astype(str)

movies["movie_id"] = movies["movie_id"].astype(str)
users["user_id"] = users["user_id"].astype(str)

print(ratings.dtypes)

user_id      object
movie_id     object
rating        int64
timestamp     int64
dtype: object


In [3]:
#preparing movie genre 
movies["genres_list"] = movies["genres"].str.split("|")

display(movies[["title", "genres", "genres_list"]].head())

,title,genres,genres_list
0,Toy Story (1995),Animation|Children's|Comedy,"[Animation, Children's, Comedy]"
1,Jumanji (1995),Adventure|Children's|Fantasy,"[Adventure, Children's, Fantasy]"
2,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]"
3,Waiting to Exhale (1995),Comedy|Drama,"[Comedy, Drama]"
4,Father of the Bride Part II (1995),Comedy,[Comedy]


In [4]:
#creating unique genre vocabulary 
all_genres = sorted(set(
    genre
    for genre_list in movies["genres_list"]
    for genre in genre_list
))

print("Total unique genres:", len(all_genres))
print(all_genres)

Total unique genres: 18
['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [5]:
#multihot encoding 
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=all_genres)

genre_encoded = mlb.fit_transform(movies["genres_list"])

genre_features = pd.DataFrame(
    genre_encoded,
    columns=mlb.classes_,
    index=movies.index
)

print("Genre feature shape:", genre_features.shape)
display(genre_features.head())

Genre feature shape: (3883, 18)


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
#combining genre features with movie ids
movie_features = pd.concat(
    [
        movies[["movie_id", "title"]],
        genre_features
    ],
    axis=1
)

print("Movie features shape:", movie_features.shape)
display(movie_features.head())

Movie features shape: (3883, 20)


,movie_id,title,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
#user demographic features 
print("Gender:", users["gender"].unique())
print("Age groups:", sorted(users["age"].unique()))
print("Occupation codes:", sorted(users["occupation"].unique()))

Gender: ['F' 'M']
Age groups: [np.int64(1), np.int64(18), np.int64(25), np.int64(35), np.int64(45), np.int64(50), np.int64(56)]
Occupation codes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]


In [8]:
#encoder gender 
users["gender_encoded"] = users["gender"].map({
    "F": 0,
    "M": 1
})

display(users[["user_id", "gender", "gender_encoded"]].head())

,user_id,gender,gender_encoded
0,1,F,0
1,2,M,1
2,3,M,1
3,4,M,1
4,5,M,1


In [9]:
#age  and occupation feature 
users["age"] = users["age"].astype(str)
users["occupation"] = users["occupation"].astype(str)

display(users[["user_id", "age", "occupation"]].head())

print("Unique age groups:", users["age"].nunique())
print("Unique occupations:", users["occupation"].nunique())

,user_id,age,occupation
0,1,1,10
1,2,56,16
2,3,25,15
3,4,45,7
4,5,25,20


Unique age groups: 7
Unique occupations: 21


In [10]:
#final user table 
user_features = users[
    ["user_id", "gender_encoded", "age", "occupation"]
].copy()

print("User features shape:", user_features.shape)
display(user_features.head())

User features shape: (6040, 4)


,user_id,gender_encoded,age,occupation
0,1,0,1,10
1,2,1,56,16
2,3,1,25,15
3,4,1,45,7
4,5,1,25,20


In [11]:
#user movie interaction
interactions = ratings[
    ["user_id", "movie_id", "rating", "timestamp"]
].copy()

print("Interactions shape:", interactions.shape)
display(interactions.head())

Interactions shape: (1000209, 4)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [12]:
#merging user and movie feature interaction 
interactions = interactions.merge(
    user_features,
    on="user_id",
    how="left"
)

print("Shape after user merge:", interactions.shape)
display(interactions.head())

Shape after user merge: (1000209, 7)


,user_id,movie_id,rating,timestamp,gender_encoded,age,occupation
0,1,1193,5,978300760,0,1,10
1,1,661,3,978302109,0,1,10
2,1,914,3,978301968,0,1,10
3,1,3408,4,978300275,0,1,10
4,1,2355,5,978824291,0,1,10


In [13]:
#merging movie features 
interactions = interactions.merge(
    movie_features,
    on="movie_id",
    how="left"
)

print("Shape after movie merge:", interactions.shape)
display(interactions.head())

Shape after movie merge: (1000209, 26)


,user_id,movie_id,rating,timestamp,gender_encoded,age,occupation,title,Action,Adventure,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1193,5,978300760,0,1,10,One Flew Over the Cuckoo's Nest (1975),0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,661,3,978302109,0,1,10,James and the Giant Peach (1996),0,0,...,0,0,0,1,0,0,0,0,0,0
2,1,914,3,978301968,0,1,10,My Fair Lady (1964),0,0,...,0,0,0,1,0,1,0,0,0,0
3,1,3408,4,978300275,0,1,10,Erin Brockovich (2000),0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,2355,5,978824291,0,1,10,"Bug's Life, A (1998)",0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
#checking for missing value  for both features 
print("Missing values in interactions:")
print(interactions.isnull().sum().sum())

print("\nColumns with missing values:")
print(interactions.isnull().sum()[interactions.isnull().sum() > 0])

Missing values in interactions:
0

Columns with missing values:
Series([], dtype: int64)


In [15]:
print("Duplicate user-movie interactions:",
      interactions.duplicated(subset=["user_id", "movie_id"]).sum())

Duplicate user-movie interactions: 0


In [16]:
#converting the timpstamps in readable form 
interactions["timestamp"] = pd.to_datetime(
    interactions["timestamp"], unit="s"
)

interactions[["user_id", "movie_id", "rating", "timestamp"]].head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,2000-12-31 22:12:40
1,1,661,3,2000-12-31 22:35:09
2,1,914,3,2000-12-31 22:32:48
3,1,3408,4,2000-12-31 22:04:35
4,1,2355,5,2001-01-06 23:38:11


In [17]:
#sorting interaction  according to  timestamps 
interactions = interactions.sort_values(
    by="timestamp"
).reset_index(drop=True)

display(interactions[["user_id", "movie_id", "rating", "timestamp"]].head())

,user_id,movie_id,rating,timestamp
0,6040,858,4,2000-04-25 23:05:32
1,6040,2384,4,2000-04-25 23:05:54
2,6040,593,5,2000-04-25 23:05:54
3,6040,1961,4,2000-04-25 23:06:17
4,6040,2019,5,2000-04-25 23:06:17


In [18]:
print("Earliest interaction:", interactions["timestamp"].min())
print("Latest interaction:", interactions["timestamp"].max())

Earliest interaction: 2000-04-25 23:05:32
Latest interaction: 2003-02-28 17:49:50


In [19]:
#interaction by year
interactions["year"] = interactions["timestamp"].dt.year

print(interactions["year"].value_counts().sort_index())

year
2000    904757
2001     68058
2002     24046
2003      3348
Name: count, dtype: int64


In [20]:
#rating distribution 
print(interactions["rating"].value_counts().sort_index())

rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64


In [21]:
#distribution of user interaction 
user_interaction_counts = interactions["user_id"].value_counts()

print(user_interaction_counts.describe())

count    6040.000000
mean      165.597517
std       192.747029
min        20.000000
25%        44.000000
50%        96.000000
75%       208.000000
max      2314.000000
Name: count, dtype: float64


In [22]:
#movie interaction count 
movie_interaction_counts = interactions["movie_id"].value_counts()

print(movie_interaction_counts.describe())

count    3706.000000
mean      269.889099
std       384.047838
min         1.000000
25%        33.000000
50%       123.500000
75%       350.000000
max      3428.000000
Name: count, dtype: float64


In [23]:
#zero interaction movies
rated_movie_ids = set(interactions["movie_id"].unique())
all_movie_ids = set(movie_features["movie_id"].unique())

unrated_movie_ids = all_movie_ids - rated_movie_ids

print("Total movies:", len(all_movie_ids))
print("Rated movies:", len(rated_movie_ids))
print("Unrated movies:", len(unrated_movie_ids))

Total movies: 3883
Rated movies: 3706
Unrated movies: 177


In [24]:
#retrival dataset 
retrieval_interactions = interactions[
    interactions["rating"] >= 4
][["user_id", "movie_id", "timestamp"]].copy()

print("Retrieval interactions shape:", retrieval_interactions.shape)
display(retrieval_interactions.head())

Retrieval interactions shape: (575281, 3)


,user_id,movie_id,timestamp
0,6040,858,2000-04-25 23:05:32
1,6040,2384,2000-04-25 23:05:54
2,6040,593,2000-04-25 23:05:54
3,6040,1961,2000-04-25 23:06:17
4,6040,2019,2000-04-25 23:06:17


In [25]:
#unique users and movies in positive interactions.
print("Unique users:", retrieval_interactions["user_id"].nunique())
print("Unique movies:", retrieval_interactions["movie_id"].nunique())

Unique users: 6038
Unique movies: 3533


In [26]:
positive_user_counts = retrieval_interactions["user_id"].value_counts()

print(positive_user_counts.describe())

count    6038.000000
mean       95.276747
std       105.005005
min         1.000000
25%        27.000000
50%        58.000000
75%       124.000000
max      1435.000000
Name: count, dtype: float64


In [27]:
#positive interaction count per user 
positive_movie_counts = retrieval_interactions["movie_id"].value_counts()

print(positive_movie_counts.describe())

count    3533.000000
mean      162.830739
std       285.798512
min         1.000000
25%        13.000000
50%        49.000000
75%       183.000000
max      2853.000000
Name: count, dtype: float64


In [28]:
#retrieval dataset duplicates
print(
    "Duplicate user-movie pairs:",
    retrieval_interactions.duplicated(
        subset=["user_id", "movie_id"]
    ).sum()
)

Duplicate user-movie pairs: 0


In [29]:
#saving retrieval dataset
os.makedirs("../processed_data", exist_ok=True)

interactions.to_parquet(
    "../processed_data/interactions.parquet",
    index=False
)

retrieval_interactions.to_parquet(
    "../processed_data/retrieval_interactions.parquet",
    index=False
)

movie_features.to_parquet(
    "../processed_data/movie_features.parquet",
    index=False
)

user_features.to_parquet(
    "../processed_data/user_features.parquet",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [32]:
print(os.listdir("../dataset/processed"))

['interactions.parquet', 'movie_features.parquet', 'retrieval_interactions.parquet', 'user_features.parquet']


In [34]:
processed_path = "../dataset/processed"